# QLoRA Summarization — Colab Training Pipeline

In [ ]:
GITHUB_REPO    = "https://github.com/ducnd58233/language-engineer.git"
GITHUB_TOKEN   = ""
HF_TOKEN       = ""
HF_REPO_ID     = "ducnd58233/qwen2.5-3b-qlora-summarization"
MAX_STEPS      = 20
N_EVAL_SAMPLES = 100

## 1. Setup

In [2]:
import os
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or GITHUB_TOKEN
except Exception:
    pass

repo_url = GITHUB_REPO.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else GITHUB_REPO

if not os.path.exists("language-engineer"):
    subprocess.run(["git", "clone", repo_url, "language-engineer"], check=True)
else:
    subprocess.run(["git", "-C", "language-engineer", "pull"], check=True)

: 

In [3]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q peft trl transformers accelerate bitsandbytes datasets \
    rouge-score sacrebleu bert-score python-dotenv pyyaml tqdm

: 

In [4]:
%cd language-engineer

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")
    f.write(f"HF_REPO_ID={HF_REPO_ID}\n")

/content/language-engineer


: 

## 2. (Optional) Mount Google Drive

Uncomment to persist checkpoints and datasets across Colab sessions.
Without this, all data is lost when the runtime disconnects.

In [5]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# os.makedirs("/content/drive/MyDrive/lang-eng/runs", exist_ok=True)
# os.makedirs("/content/drive/MyDrive/lang-eng/datasets", exist_ok=True)
# if not os.path.exists("runs"):
#     os.symlink("/content/drive/MyDrive/lang-eng/runs", "runs")
# if not os.path.exists("datasets"):
#     os.symlink("/content/drive/MyDrive/lang-eng/datasets", "datasets")

: 

## 3. Prepare Datasets

In [ ]:
!python scripts/prepare_datasets.py

: 

In [10]:
!python scripts/process_datasets.py

Datasets found: ['cnn', 'xsum']
Generating train split: 287113 examples [00:59, 4808.66 examples/s] 
Generating train split: 204017 examples [00:11, 18488.60 examples/s]
Generating train split: 13368 examples [00:00, 68340.63 examples/s]
Generating train split: 11327 examples [00:00, 100917.61 examples/s]
Generating train split: 11490 examples [00:00, 62821.31 examples/s]
Generating train split: 11333 examples [00:00, 74772.10 examples/s]
[leakage] removed 10 rows from validation found in train
[leakage] removed 9 rows from test found in train

Split            Raw     Hard      IQR    Dedup    Final
----------------------------------------------------------------
hard filter: 100% 491130/491130 [00:21<00:00, 22663.99 examples/s]
train        491,130  -10,500  -28,576  -3,050  449,004
format prompts: 100% 449004/449004 [01:26<00:00, 5210.25 examples/s]
Creating parquet from Arrow format: 100% 450/450 [00:47<00:00,  9.41ba/s]
Saved train -> /content/language-engineer/datasets/processed/

: 

## 4. Train

In [ ]:
!python scripts/train.py --max-steps {MAX_STEPS}

: 

## 5. Evaluate

Compares base model (zero-shot) vs. fine-tuned adapter on the test set.
Results saved to `results/eval_results.json`.

In [ ]:
!python scripts/evaluate.py --n-samples {N_EVAL_SAMPLES}

: 

## 6. Upload to HuggingFace Hub

In [ ]:
!python scripts/hub.py upload \
    --adapter runs/Qwen2.5-3B-Instruct_qlora/final \
    --repo {HF_REPO_ID}

: 

## 7. (Optional) Download from HuggingFace Hub

Pull a previously uploaded adapter back to local disk (e.g. to resume eval on a new runtime).

In [ ]:
!python scripts/hub.py download \
    --repo {HF_REPO_ID} \
    --output runs/downloaded

: 